In [1]:
!pip install pymupdf

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
import fitz  # PyMuPDF

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

pdf_path = r"C:/Users/preet/dataset_upi.pdf"
raw_text = extract_pdf_text(pdf_path)

# Show first few lines
print(raw_text[:1000])

STATEMENT OF ACCOUNT
CUSTOMER ID                     :    10******93
ACCOUNT NO                      :    10001417906
STATEMENT PERIOD          :    2026-04-01 TO 2026-09-30
CUSTOMER NAME
:
Mr. Aravind Raj J
COMMUNICATION
ADDRESS:
:
S/O:
Jaishankar,23/57,DEMELLOWS
ROAD,CHE NNAI
DISTRICT,Perambur
Barracks,Chennai .
Chennai 600012 TAMIL
NADU INDIA
EMAIL ID
:
e*********3@gmail.com
PHONE NO
:
********4538
CKYC ID
:
**********4500
NOMINATION
:
Registered
NOMINEE NAME
:
Sheela J
ACCOUNT BRANCH
:
Nungambakkam Branch
BRANCH ADDRESS
:
Tahera Manor,
Nungambakkam High
Road Gf and first floor
Chennai 600034
IFSC
:
IDFB0080102
MICR
:
600751002
ACCOUNT OPENING
DATE
:
2016-11-18
ACCOUNT STATUS
:
ACTIVE
ACCOUNT TYPE
:
Corporate Salary
Platinum
CURRENCY
:
INR
Opening Balance
Total Debit
Total Credit
Closing Balance
60,325.08
6,72,150.89
6,11,936.63
110.83
Transaction
Date
Value Date
Particulars
Cheque
No
Debit
Credit
Balance
Opening Balance
60,325.08
01-Apr-2026
01-Apr-2026
UPI/DR/609168468659/
Surya M

In [4]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    all_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                all_text += text + "\n"
    return all_text

pdf_file = r"C:/Users/preet/dataset_upi.pdf" # raw string to avoid warnings

raw_text = extract_text_from_pdf(pdf_file)

with open("extracted_statement.txt", "w", encoding="utf-8") as f:
    f.write(raw_text)

print("PDF text extracted successfully.")

PDF text extracted successfully.


In [5]:
import csv
import re

input_file = r'C:/Users/preet/pfgu/extracted_statement.txt'
output_file = r'C:/Users/preet/pfgu/transactions.csv'

# Read the extracted statement
with open(input_file, 'r', encoding='utf-8') as f:
    lines = f.readlines()

transactions = []
current_transaction = None

# Date format in your statement: 01-Apr-2026
date_pattern = re.compile(
    r'^(\d{2}-[A-Za-z]{3}-\d{4})\s+'
    r'(\d{2}-[A-Za-z]{3}-\d{4})\s+'
)

# Amount format: 40.00, 6,72,150.89, etc.
amount_pattern = re.compile(
    r'\d[\d,]*\.\d{2}'
)

for line in lines:

    line = line.strip()

    if not line:
        continue

    # Check whether line starts with transaction date
    date_match = date_pattern.match(line)

    if date_match:

        # Save previous transaction
        if current_transaction:
            transactions.append(current_transaction)

        # Get dates
        transaction_date = date_match.group(1)
        value_date = date_match.group(2)

        # Remove the two dates
        remaining = line[date_match.end():].strip()

        # Find amounts
        amounts = amount_pattern.findall(remaining)

        # Remove amounts from the text
        details = remaining

        for amount in amounts:
            details = details.replace(amount, '', 1)

        # Clean details
        details = re.sub(r'\s+', ' ', details).strip()

        # Default values
        debit = ''
        credit = ''
        balance = ''

        # Your statement has:
        # Particulars | Debit | Credit | Balance
        #
        # Therefore the LAST amount is Balance.
        # If there are TWO amounts:
        # first = Debit/Credit
        # second = Balance

        if len(amounts) == 1:
            balance = amounts[0]

        elif len(amounts) == 2:
            debit = amounts[0]
            balance = amounts[1]

        elif len(amounts) >= 3:
            debit = amounts[-3]
            credit = amounts[-2]
            balance = amounts[-1]

        current_transaction = {
            'Date': transaction_date,
            'Value Date': value_date,
            'Transaction Details': details,
            'Debit': debit,
            'Credit': credit,
            'Balance': balance
        }

    else:
        # This is a continuation of the transaction details
        if current_transaction:

            # Don't add obvious non-transaction headings
            if not line.startswith(
                ('Opening Balance', 'Transaction Cheque', 'Value Date')
            ):
                current_transaction['Transaction Details'] += ' ' + line


# Add final transaction
if current_transaction:
    transactions.append(current_transaction)


# Clean transaction details
for transaction in transactions:
    transaction['Transaction Details'] = re.sub(
        r'\s+',
        ' ',
        transaction['Transaction Details']
    ).strip()


# Write CSV
with open(
    output_file,
    'w',
    newline='',
    encoding='utf-8'
) as csvfile:

    fieldnames = [
        'Date',
        'Value Date',
        'Transaction Details',
        'Debit',
        'Credit',
        'Balance'
    ]

    writer = csv.DictWriter(
        csvfile,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(transactions)


print("Success!")
print(f"Extracted {len(transactions)} transactions")
print(f"CSV saved to: {output_file}")

Success!
Extracted 649 transactions
CSV saved to: C:/Users/preet/pfgu/transactions.csv


In [7]:
df = pd.read_csv("transactions.csv")

In [9]:
def clean_currency(col):
    return (
        col.astype(str)
           .str.replace('INR', '', regex=False)
           .str.replace(',', '', regex=False)
           .str.strip()
           .replace('', '0')
           .astype(float)
    )

df['Debit'] = clean_currency(df['Debit'])
df['Credit'] = clean_currency(df['Credit'])
df['Balance'] = clean_currency(df['Balance'])

In [28]:
import pandas as pd

df = pd.read_csv(r'C:/Users/preet/pfgu/transactions.csv')

df['Debit'] = pd.to_numeric(df['Debit'], errors='coerce').fillna(0)
df['Credit'] = pd.to_numeric(df['Credit'], errors='coerce').fillna(0)

print("Largest Debit transactions:")
print(df.nlargest(20, 'Debit')[[
    'Date', 'Transaction Details', 'Debit', 'Credit', 'Balance'
]].to_string(index=False))

print("\nLargest Credit transactions:")
print(df.nlargest(20, 'Credit')[[
    'Date', 'Transaction Details', 'Debit', 'Credit', 'Balance'
]].to_string(index=False))

Largest Debit transactions:
       Date                               Transaction Details  Debit  Credit   Balance
13-Jun-2026                        Google I/UTIB/gpay-ut/ UPI 957.50     0.0  5,270.35
14-Jul-2026                        Google I/UTIB/gpay-ut/ UPI 957.50     0.0 19,582.61
04-May-2026                        Google I/UTIB/gpay-ut/ UPI 928.50     0.0 63,289.34
15-Aug-2026                         revathi4/INDB/revathi/UPI 920.00     0.0 13,584.96
06-May-2026                        ZOMATO L/UTIB/ zomatoo/UPI 909.74     0.0 25,223.60
03-Apr-2026                        DUMMY NA/ ciub/5001010/UPI 900.00     0.0 55,359.95
31-May-2026                                  ciub/5001010/UPI 900.00     0.0 45,288.26
12-May-2026                        CONFIRM /UTIB/confirm/ UPI 896.00     0.0 22,544.36
08-Sep-2026 AIRTEL P/INDB/airtelp/ AirtelBroadbandBillPayment 879.22     0.0  5,052.77
08-Aug-2026                        ORBGEN T/HDFC/distric/ UPI 872.08     0.0 12,214.44
03-Apr-2026    

In [29]:
import pandas as pd

df = pd.read_csv(r'C:/Users/preet/pfgu/transactions.csv')

df['Debit'] = pd.to_numeric(df['Debit'], errors='coerce').fillna(0)
df['Credit'] = pd.to_numeric(df['Credit'], errors='coerce').fillna(0)

print("Rows:", len(df))
print("Debit:", df['Debit'].sum())
print("Credit:", df['Credit'].sum())

print("\nDebit count:", (df['Debit'] > 0).sum())
print("Credit count:", (df['Credit'] > 0).sum())

print("\nFirst balance:", df.iloc[0]['Balance'])
print("Last balance:", df.iloc[-1]['Balance'])

Rows: 649
Debit: 99201.04000000001
Credit: 14105.12

Debit count: 405
Credit count: 35

First balance: 60,285.08
Last balance: 110.83


In [30]:
print(df[['Date', 'Debit', 'Credit', 'Balance']].tail(20).to_string(index=False))

       Date  Debit  Credit  Balance
16-Sep-2026  210.0     0.0 1,232.83
17-Sep-2026  100.0     0.0 1,132.83
17-Sep-2026    0.0     1.0 1,133.83
18-Sep-2026  190.0     0.0   943.83
19-Sep-2026  130.0     0.0   813.83
20-Sep-2026   60.0     0.0   753.83
22-Sep-2026  120.0     0.0   633.83
22-Sep-2026  200.0     0.0   433.83
22-Sep-2026  225.0     0.0   208.83
23-Sep-2026    0.0   900.0 1,108.83
23-Sep-2026    0.0     0.0   110.83
24-Sep-2026  120.0     0.0   833.83
24-Sep-2026   10.0     0.0   823.83
24-Sep-2026  300.0     0.0   523.83
25-Sep-2026  100.0     0.0   423.83
26-Sep-2026  200.0     0.0   223.83
26-Sep-2026    0.0     0.0 3,723.83
27-Sep-2026    0.0     0.0   223.83
29-Sep-2026  120.0     0.0   103.83
30-Sep-2026    0.0     7.0   110.83


Debit = Previous Balance - Current Balance
Credit = Current Balance - Previous Balance

In [31]:
import csv
import re

input_file = r'C:/Users/preet/pfgu/extracted_statement.txt'
output_file = r'C:/Users/preet/pfgu/transactions.csv'

# Read extracted text
with open(input_file, 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Date format in your statement
date_pattern = re.compile(
    r'^(\d{2}-[A-Za-z]{3}-\d{4})\s+'
    r'(\d{2}-[A-Za-z]{3}-\d{4})\s+'
)

# Amount format
amount_pattern = re.compile(r'\d[\d,]*\.\d{2}')

transactions = []
current = None
pending_direction = None


def save_transaction():
    global current

    if current is None:
        return

    text = current['raw_text']

    # Find all amounts in this transaction
    amounts = amount_pattern.findall(text)

    # Need at least transaction amount + balance
    if len(amounts) < 2:
        return

    # Last amount = balance
    balance = amounts[-1]

    # Amount immediately before balance = transaction amount
    transaction_amount = amounts[-2]

    # Remove amounts from transaction details
    details = text

    for amount in amounts:
        details = details.replace(amount, '', 1)

    details = re.sub(r'\s+', ' ', details).strip()

    debit = ''
    credit = ''

    direction = current['direction']

    # First use DR / CR marker
    if direction == 'DR':
        debit = transaction_amount

    elif direction == 'CR':
        credit = transaction_amount

    else:
        # For non-UPI transactions, use balance movement
        previous_balance = current['previous_balance']

        if previous_balance is not None:

            old_balance = previous_balance
            new_balance = float(balance.replace(',', ''))
            amount = float(transaction_amount.replace(',', ''))

            # Balance decreased = Debit
            if abs(old_balance - amount - new_balance) < 0.01:
                debit = transaction_amount

            # Balance increased = Credit
            elif abs(old_balance + amount - new_balance) < 0.01:
                credit = transaction_amount

    transactions.append({
        'Date': current['date'],
        'Value Date': current['value_date'],
        'Transaction Details': details,
        'Debit': debit,
        'Credit': credit,
        'Balance': balance
    })


# Process the text
for line in lines:

    line = line.strip()

    if not line:
        continue

    upper_line = line.upper()

    # Ignore repeated headers / footers
    if line.startswith('REGISTERED OFFICE:'):
        continue

    if line.startswith('STATEMENT OF ACCOUNT'):
        continue

    if line.startswith('CUSTOMER ID'):
        continue

    if line.startswith('ACCOUNT NO'):
        continue

    if line.startswith('STATEMENT PERIOD'):
        continue

    if line.startswith('Opening Balance Total Debit'):
        continue

    if line.startswith('Transaction Cheque'):
        continue

    if line.startswith('Date No'):
        continue

    if line.startswith('Page '):
        continue

    # Detect UPI debit/credit marker
    if 'UPI/DR/' in upper_line:
        pending_direction = 'DR'
        continue

    if 'UPI/CR/' in upper_line:
        pending_direction = 'CR'
        continue

    # Check for transaction date
    date_match = date_pattern.match(line)

    if date_match:

        # Save previous transaction
        save_transaction()

        transaction_date = date_match.group(1)
        value_date = date_match.group(2)

        remaining_text = line[date_match.end():].strip()

        # Previous transaction balance
        previous_balance = None

        if transactions:
            previous_balance = float(
                transactions[-1]['Balance'].replace(',', '')
            )

        current = {
            'date': transaction_date,
            'value_date': value_date,
            'raw_text': remaining_text,
            'direction': pending_direction,
            'previous_balance': previous_balance
        }

        pending_direction = None

    else:

        # Continuation line belongs to current transaction
        if current is not None:
            current['raw_text'] += ' ' + line


# Save final transaction
save_transaction()


# Write CSV
with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:

    fieldnames = [
        'Date',
        'Value Date',
        'Transaction Details',
        'Debit',
        'Credit',
        'Balance'
    ]

    writer = csv.DictWriter(
        csvfile,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(transactions)


print("Success!")
print("Number of transactions:", len(transactions))
print("CSV saved to:", output_file)

Success!
Number of transactions: 649
CSV saved to: C:/Users/preet/pfgu/transactions.csv


In [32]:
import pandas as pd

csv_file = r'C:/Users/preet/pfgu/transactions.csv'

df = pd.read_csv(csv_file)

def clean_currency(col):
    return (
        col.fillna('')
           .astype(str)
           .str.replace(',', '', regex=False)
           .str.strip()
           .replace('', '0')
           .astype(float)
    )

df['Debit'] = clean_currency(df['Debit'])
df['Credit'] = clean_currency(df['Credit'])
df['Balance'] = clean_currency(df['Balance'])

print("Number of transactions:", len(df))
print("Total Debit:", df['Debit'].sum())
print("Total Credit:", df['Credit'].sum())
print("Opening/First Balance:", df.iloc[0]['Balance'])
print("Closing/Last Balance:", df.iloc[-1]['Balance'])

Number of transactions: 649
Total Debit: 26310913.78
Total Credit: 3023781.16
Opening/First Balance: 60285.08
Closing/Last Balance: 110.83


In [33]:
df.head(50)

,Date,Value Date,Transaction Details,Debit,Credit,Balance
0,01-Apr-2026,01-Apr-2026,Surya Mi/YESB/paytmqr/ UPI,40.00,0.00,60285.08
1,01-Apr-2026,01-Apr-2026,Google P/utib/playsto/ Mandate,449.00,0.00,59836.08
2,02-Apr-2026,02-Apr-2026,VINOTH K/HDFC/sridhar/ UPI,500.00,0.00,59336.08
3,02-Apr-2026,02-Apr-2026,RAJESH /IOBA/rajeshp/ UPI,120.00,0.00,59216.08
4,02-Apr-2026,02-Apr-2026,ADITHYA /HDFC/adithya/ UPI Value Date Particul...,611936.63,0.00,110.83
5,02-Apr-2026,02-Apr-2026,ADITHYA /SBIP/sbibhim/ Static,210.00,0.00,58706.08
6,02-Apr-2026,02-Apr-2026,MOHAMED /BKID/ mohamed/UPI,130.00,0.00,58576.08
7,02-Apr-2026,02-Apr-2026,ZOMATO L/UTIB/ zomatoo/UPI,414.33,0.00,58161.75
8,03-Apr-2026,03-Apr-2026,Google I/UTIB/gpayrec/ UPI,220.90,0.00,57940.85
9,03-Apr-2026,03-Apr-2026,Euronet /UTIB/gpayrec/ UPI,860.90,0.00,57079.95


In [34]:
def categorize(description):
    desc = str(description).lower()

    # Food Delivery
    if "swiggy" in desc or "zomato" in desc or "blinkit" in desc:
        return "Food Delivery"

    # Grocery Delivery
    elif "zepto" in desc:
        return "Grocery Delivery"

    # Travel
    elif ("redbus" in desc or "uber" in desc or "ola" in desc
          or "aswini automobiles" in desc
          or "suncityfuels" in desc
          or "zingbus" in desc):
        return "Travel"

    # Recharge
    elif ("jio" in desc or "recharge" in desc
          or "bharti airtel limited" in desc):
        return "Recharge"

    # Broadband Bill
    elif ("airtel broadband" in desc
          or "airtelbroadbandbillpayment" in desc):
        return "Broadband Bill"

    # Insurance
    elif "insurance" in desc:
        return "Insurance"

    # Amazon Refund
    elif "amazon" in desc and "refund" in desc:
        return "Amazon Refund"

    # Online Shopping
    elif ("amazon" in desc or "flipkart" in desc
          or "myntra" in desc or "ajio" in desc
          or "meesho" in desc or "superssmart" in desc):
        return "Online Shopping"

    # Healthcare
    elif "life science" in desc:
        return "Healthcare"

    # Groceries
    elif "grand fresh" in desc or "bigbasket" in desc:
        return "Groceries"

    # Utilities
    elif "maintenance" in desc:
        return "Utilities"

    # Google Play / Google
    elif "google play" in desc or "google" in desc:
        return "Google Payment"

    # Google Play Mandate
    elif "mandate" in desc:
        return "Recurring / Mandate Payment"

    # PhonePe
    elif "phonepe" in desc:
        return "UPI Payment"

    # Paytm
    elif "paytm" in desc:
        return "UPI Payment"

    # BharatPe
    elif "bharatpe" in desc:
        return "Merchant Payment"

    # UPI Cash Withdrawal
    elif "upi cash withdrawal" in desc or "cash withdrawal" in desc:
        return "Cash Withdrawal"

    # NEFT Salary
    elif "neft" in desc and "salary" in desc:
        return "Salary / Income"

    # Monthly Savings Interest
    elif "monthly savings" in desc or "interest credit" in desc:
        return "Interest Income"

    # IFT Transfer
    elif "ift" in desc:
        return "Bank Transfer"

    # UPI Reversal
    elif "upi reversal" in desc or "reversal" in desc:
        return "UPI Reversal"

    # UPI Refund
    elif "upi refund" in desc or "refund" in desc:
        return "UPI Refund"

    # Story TV recurring payment
    elif "story tv" in desc or "storytv" in desc:
        return "Subscription"

    # PVR / INOX
    elif "pvr" in desc or "inox" in desc:
        return "Entertainment"

    # Mobikwik
    elif "mobikwik" in desc:
        return "Wallet / Payment Service"

    # Other UPI transactions
    elif "upi" in desc:
        return "Other UPI Merchant Payment"

    # Everything else
    else:
        return "Others"


# Apply category
df["Category"] = df["Transaction Details"].apply(categorize)


# Save categorized dataframe
df.to_csv(
    r"C:/Users/preet/pfgu/categorized_transactions_clean.csv",
    index=False
)

print("Categorization completed!")

print("\nCategory counts:")
print(df["Category"].value_counts())

print("\nFirst 10 rows:")
print(df.head(10))

Categorization completed!

Category counts:
Category
Other UPI Merchant Payment     399
UPI Payment                     94
Others                          51
Google Payment                  39
Food Delivery                   19
Online Shopping                 11
Travel                           7
Grocery Delivery                 7
Bank Transfer                    6
Broadband Bill                   3
Wallet / Payment Service         2
Recharge                         2
Amazon Refund                    2
Interest Income                  2
Recurring / Mandate Payment      1
Subscription                     1
Entertainment                    1
Merchant Payment                 1
Insurance                        1
Name: count, dtype: int64

First 10 rows:
          Date   Value Date  \
0  01-Apr-2026  01-Apr-2026   
1  01-Apr-2026  01-Apr-2026   
2  02-Apr-2026  02-Apr-2026   
3  02-Apr-2026  02-Apr-2026   
4  02-Apr-2026  02-Apr-2026   
5  02-Apr-2026  02-Apr-2026   
6  02-Apr-2026  02-Apr-

In [36]:
import pandas as pd

# Split Transaction Details and get the Merchant part
df['Merchant'] = df['Transaction Details'].str.split('/', n=3).str[1]

# Clean Merchant column
df['Merchant'] = df['Merchant'].fillna('').str.strip().str.lower()


# Categorize transactions
def categorize(merchant):
    
    if not merchant:
        return "Others"

    # Food
    if "swiggy" in merchant or "zomato" in merchant:
        return "Food Delivery"

    # Grocery delivery
    elif "zepto" in merchant or "blinkit" in merchant:
        return "Grocery Delivery"

    # Amazon refund must come before Amazon shopping
    elif "amazon" in merchant and "refund" in merchant:
        return "Amazon Refund"

    # Online shopping
    elif ("amazon" in merchant or 
          "flipkart" in merchant or 
          "myntra" in merchant or 
          "ajio" in merchant or 
          "v mart" in merchant or 
          "meesho" in merchant or 
          "superssmart" in merchant):
        return "Online Shopping"

    # Google payments
    elif "google play" in merchant and "mandate" in merchant:
        return "Recurring / Mandate Payment"

    elif "google play" in merchant or "google" in merchant:
        return "Google Payment"

    # Broadband
    elif "airtel broadband" in merchant:
        return "Broadband Bill"

    # Recharge
    elif ("jio" in merchant or 
          "recharge" in merchant or 
          "bharti airtel" in merchant):
        return "Recharge"

    # Insurance
    elif ("insurance" in merchant or 
          "star health" in merchant):
        return "Insurance"

    # Healthcare
    elif "life science" in merchant:
        return "Healthcare"

    # Groceries
    elif ("grand fresh" in merchant or 
          "bigbasket" in merchant):
        return "Groceries"

    # Travel
    elif ("redbus" in merchant or 
          "uber" in merchant or 
          "ola" in merchant or 
          "aswini automobiles" in merchant or 
          "suncityfuels" in merchant or
          "zingbus" in merchant):
        return "Travel"

    # Entertainment
    elif "pvr" in merchant or "inox" in merchant:
        return "Entertainment"

    # Subscription
    elif "story tv" in merchant or "storytv" in merchant:
        return "Subscription"

    # Wallet / payment service
    elif "mobikwik" in merchant:
        return "Wallet / Payment Service"

    # BharatPe
    elif "bharatpe" in merchant:
        return "Merchant Payment"

    # Utilities
    elif "maintenance" in merchant:
        return "Utilities"

    # Otherwise
    else:
        return "Others"


# Apply categorization
df['Category'] = df['Merchant'].apply(categorize)


# Check the result
print(df[['Merchant', 'Category']].head(20))

   Merchant Category
0      yesb   Others
1      utib   Others
2      hdfc   Others
3      ioba   Others
4      hdfc   Others
5      sbip   Others
6      bkid   Others
7      utib   Others
8      utib   Others
9      utib   Others
10     idib   Others
11     ciub   Others
12     cnrb   Others
13     ucba   Others
14     tmbl   Others
15     yesb   Others
16     yesb   Others
17     utib   Others
18     yesb   Others
19     hdfc   Others


In [38]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

df["Month"] = df["Date"].dt.to_period("M")

monthly_spending = df.groupby("Month")["Debit"].sum()

print("📆 Monthly Spending:\n")
print(monthly_spending)

📆 Monthly Spending:

Month
2026-04    4393781.50
2026-05    5019758.73
2026-06    3727318.09
2026-07    4989808.42
2026-08    5660169.56
2026-09    2520077.48
Freq: M, Name: Debit, dtype: float64


In [40]:
category_spending = df.groupby("Category")["Debit"].sum().sort_values(ascending=False)

print("🏷 Category-wise Spending:\n")
print(category_spending)

🏷 Category-wise Spending:

Category
Others             25697001.21
Online Shopping      613812.57
Recharge                100.00
Name: Debit, dtype: float64


In [41]:
!pip install openai

In [60]:
import requests

api_key = "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"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(
    "https://openrouter.ai/api/v1/auth/key",
    headers=headers
)

print("Status:", response.status_code)
print(response.text)

Status: 200
{"data":{"label":"sk-or-v1-38c...8c7","is_management_key":false,"is_provisioning_key":false,"limit":100,"limit_reset":null,"limit_remaining":100,"include_byok_in_limit":false,"usage":0,"usage_daily":0,"usage_weekly":0,"usage_monthly":0,"byok_usage":0,"byok_usage_daily":0,"byok_usage_weekly":0,"byok_usage_monthly":0,"is_free_tier":true,"expires_at":"2027-04-06T07:24:31.483Z","creator_user_id":"user_3KP1D1CqVhxkYFQwVtjIpyKeyWQ","organization_id":null,"workspace_id":"f7a7fcd0-cc80-5af9-b38a-3985379f51d5","allowed_data_regions":["global"],"free_model_daily_requests":{"used":0,"limit":50,"remaining":50},"rate_limit":{"requests":-1,"interval":"10s","note":"This field is deprecated and safe to ignore."}}}


In [61]:
import requests

api_key = "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"

monthly_str = "\n".join(
    [f"{month}: ₹{amt:,.2f}" for month, amt in monthly_spending.items()]
)

category_str = "\n".join(
    [f"{cat}: ₹{amt:,.2f}" for cat, amt in category_spending.items()]
)

prompt = f"""
You are a personal finance advisor.

Analyze ONLY the following financial data.
Do not invent any numbers.

Monthly Spending:
{monthly_str}

Category-wise Spending:
{category_str}

Please provide:

1. A short summary of the spending pattern.
2. The highest spending category.
3. The highest spending month.
4. Three practical ways to reduce unnecessary spending.
5. Three simple budgeting or savings tips.
6. A short disclaimer saying this is general financial guidance.

Use Indian Rupees (₹).
Do not assume income or other financial information that is not provided.
"""

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

payload = {
    "model": "openrouter/auto",
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "temperature": 0.2
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

print("Status:", response.status_code)

if response.status_code == 200:
    advice = response.json()["choices"][0]["message"]["content"]

    print("\n💡 Personalized Financial Advice:\n")
    print(advice)

else:
    print("❌ Error:", response.status_code)
    print(response.text)

Status: 200

💡 Personalized Financial Advice:

### 1. Spending pattern
Spending varied across the six months, ranging from ₹2,520,077.48 in September to ₹5,660,169.56 in August. It peaked in August, then fell sharply in September. The listed monthly amounts total ₹26,310,913.78.

### 2. Highest spending category
**Others — ₹25,697,001.21**, by a wide margin. Reviewing what is included in this category could help identify specific areas to manage.

### 3. Highest spending month
**August 2026 — ₹5,660,169.56.**

### 4. Three practical ways to reduce unnecessary spending
- Review transactions in **Others** and separate them into clearer subcategories.
- Before online purchases, compare prices and pause to check whether each item is needed.
- Look over spending after higher-spend months to spot purchases or habits you may want to limit.

### 5. Three simple budgeting or savings tips
- Track spending by month and category so changes are easy to notice.
- Set category limits based on your ow

In [62]:
import requests

api_key = "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"

monthly_str = "\n".join(
    [f"{month}: ₹{amt:,.2f}" for month, amt in monthly_spending.items()]
)

category_str = "\n".join(
    [f"{cat}: ₹{amt:,.2f}" for cat, amt in category_spending.items()]
)

prompt = f"""
You are a personal finance advisor.

Analyze the following financial data:

Monthly Spending:
{monthly_str}

Category-wise Spending:
{category_str}

Generate personalized financial advice in this style:

**Personalized Financial Tips:**

1. **Control Unnecessary Spending**
   Give a short practical suggestion based on the user's spending pattern.

2. **Reduce Spending in High-Cost Categories**
   Identify the important spending categories using WORDS ONLY.
   Do not mention any amounts, percentages, or numerical values.

3. **Improve Saving Habits**
   Give simple and practical suggestions for better budgeting and saving.

**Key Takeaway:**
Give a short overall conclusion about the user's spending habits
and what they can do to improve them.

IMPORTANT:
- Do NOT show any numbers.
- Do NOT show amounts.
- Do NOT show percentages.
- Do NOT show dates or months.
- Do NOT calculate or mention numerical values.
- Use category names when useful.
- Use simple, easy-to-understand language.
- Do not assume the user's income.
- Do not recommend specific investment products.
- Give general financial guidance only.
"""

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

payload = {
    "model": "openrouter/auto",
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "temperature": 0.2
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

print("Status:", response.status_code)

if response.status_code == 200:

    advice = response.json()["choices"][0]["message"]["content"]

    print("\n💡 Financial Advice:\n")
    print(advice)

else:
    print("❌ Error:", response.status_code)
    print(response.text)

Status: 200

💡 Financial Advice:

**Personalized Financial Tips:**

1. **Control Unnecessary Spending**  
   A large part of your spending is grouped under “Others,” which makes it hard to see where your money actually goes. Start by listing every expense in that category and ask yourself whether each one was truly necessary. Before making any similar purchase in the future, pause and consider if it adds real value.

2. **Reduce Spending in High-Cost Categories**  
   Focus on the “Others” category, as it is your biggest spending area. Also keep “Online Shopping” in check by avoiding impulse purchases. “Recharge” is not a major concern, so your main effort should go toward bringing structure to the less-defined categories.

3. **Improve Saving Habits**  
   Create a simple monthly budget before the month begins. Decide how much you want to save first, then plan your spending around what remains. Put savings aside as soon as you receive money, and review your spending regularly to catch

In [63]:
import requests

api_key = "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"

monthly_str = "\n".join(
    [f"{month}: ₹{amt:,.2f}" for month, amt in monthly_spending.items()]
)

category_str = "\n".join(
    [f"{cat}: ₹{amt:,.2f}" for cat, amt in category_spending.items()]
)

prompt = f"""
You are a personal finance advisor.

Analyze the user's actual spending data below.

MONTHLY SPENDING:
{monthly_str}

CATEGORY-WISE SPENDING:
{category_str}

Create a detailed and personalized financial analysis.

Follow this EXACT structure:

**Personalized Financial Tips:**

1. **[A meaningful title related to the spending pattern]**
   - Compare spending between different months when useful.
   - Calculate percentage increases or decreases when possible.
   - Mention the actual ₹ amounts.
   - Identify unusual or high-spending months.
   - Give a practical budgeting recommendation.

2. **[A meaningful title about high spending categories]**
   - Identify the highest spending categories.
   - Mention their actual ₹ amounts.
   - Calculate percentages of total spending when useful.
   - Explain what the user can do to reduce unnecessary spending.
   - Give practical examples such as meal planning, reducing shopping,
     comparing prices, or setting spending limits.

3. **[A meaningful title about savings or recurring expenses]**
   - Identify categories where savings may be possible.
   - Mention actual ₹ amounts where useful.
   - Suggest realistic ways to reduce expenses.
   - Suggest how the saved money could be allocated toward savings
     or an emergency fund.
   - Do not assume the user's income.

**Key Takeaway:**
Give a concise conclusion using the actual spending data.
Mention the most important spending pattern and the potential
area where the user can save money.

IMPORTANT RULES:

- Use the actual numbers from the supplied data.
- Calculate percentages correctly.
- Do not invent spending amounts.
- Do not invent months or categories.
- Compare months only when the required data is available.
- Mention ₹ amounts where useful.
- Include percentages where useful.
- Give specific, practical recommendations.
- Do not assume the user's income.
- Do not make guaranteed financial claims.
- Do not recommend specific stocks or risky investments.
- Keep the explanation easy to understand.
- Use bullet points inside each section when appropriate.
"""

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

payload = {
    "model": "openrouter/auto",
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "temperature": 0.2
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

print("Status:", response.status_code)

if response.status_code == 200:

    advice = response.json()["choices"][0]["message"]["content"]

    print("\n💡 Financial Advice:\n")
    print(advice)

else:
    print("❌ Error:", response.status_code)
    print(response.text)

Status: 200

💡 Financial Advice:

**Personalized Financial Tips:**

1. **Your Monthly Spending Fluctuates Sharply — Smooth It Out**
   - Your spending over six months (Apr–Sep 2026) totals roughly ₹2.63 crore, averaging about ₹43.85 lakh per month.
   - August 2026 was your highest month at ₹5,660,169.56, while September 2026 dropped to ₹2,520,077.48 — a decrease of about 55.5% from August.
   - May (₹5,019,758.73) and July (₹4,989,808.42) were also elevated, while June (₹3,727,318.09) was comparatively moderate.
   - This swing of over ₹31 lakh between your highest and lowest months suggests irregular or lumpy expenses. A practical step: set a monthly spending cap (e.g., based on your lowest months, around ₹37–40 lakh) and spread large purchases across months instead of clustering them.

2. **"Others" Dominates Your Spending — Break It Down**
   - The "Others" category accounts for ₹25,697,001.21, which is about 97.7% of your total spending of roughly ₹2.63 crore.
   - Online Shopping

In [68]:
# ============================================================
# UPI TRANSACTION CATEGORY PREDICTION
# 4 MACHINE LEARNING / DEEP LEARNING MODELS
#
# 1. Logistic Regression
# 2. Multinomial Naive Bayes
# 3. Random Forest
# 4. LSTM
# ============================================================

import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical


# ============================================================
# 1. LOAD DATASET
# ============================================================

file_path = "transactions.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Total transactions:", len(df))

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. CHECK REQUIRED COLUMN
# ============================================================

if "Transaction Details" not in df.columns:
    raise ValueError(
        "Column 'Transaction Details' not found in CSV."
    )


# ============================================================
# 3. CLEAN TRANSACTION DETAILS
# ============================================================

df["Transaction Details"] = (
    df["Transaction Details"]
    .fillna("")
    .astype(str)
)


def clean_text(text):

    text = text.lower()

    # Remove numbers
    text = re.sub(r"\d+", " ", text)

    # Remove special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["Cleaned_Text"] = df["Transaction Details"].apply(
    clean_text
)


# ============================================================
# 4. CREATE TRANSACTION CATEGORIES
# ============================================================

def assign_category(text):

    text = text.lower()

    # --------------------------------------------------------
    # FOOD
    # --------------------------------------------------------

    if any(word in text for word in [
        "zomato",
        "swiggy",
        "food",
        "zepto",
        "bigbasket",
        "dominos",
        "pizza",
        "kfc",
        "mcdonald",
        "burger king",
        "starbucks",
        "blinkit",
        "instamart"
    ]):

        return "Food"


    # --------------------------------------------------------
    # TRAVEL
    # --------------------------------------------------------

    elif any(word in text for word in [
        "uber",
        "ola",
        "rapido",
        "travel",
        "zingbus",
        "redbus",
        "fuel",
        "petrol",
        "diesel",
        "automobile",
        "irctc",
        "railway",
        "metro"
    ]):

        return "Travel"


    # --------------------------------------------------------
    # SHOPPING
    # --------------------------------------------------------

    elif any(word in text for word in [
        "amazon",
        "flipkart",
        "meesho",
        "myntra",
        "shopping",
        "v mart",
        "dmart",
        "croma",
        "reliance digital",
        "ajio",
        "tata cliq",
        "snapdeal"
    ]):

        return "Shopping"


    # --------------------------------------------------------
    # ENTERTAINMENT
    # --------------------------------------------------------

    elif any(word in text for word in [
        "netflix",
        "spotify",
        "hotstar",
        "cinema",
        "cinemas",
        "pvr",
        "inox",
        "story tv",
        "storytv",
        "bookmyshow",
        "movie"
    ]):

        return "Entertainment"


    # --------------------------------------------------------
    # BILLS
    # --------------------------------------------------------

    elif any(word in text for word in [
        "electricity",
        "water bill",
        "gas bill",
        "electric bill",
        "broadband",
        "airtel broadband",
        "maintenance",
        "bescom",
        "tneb",
        "tangedco"
    ]):

        return "Bills"


    # --------------------------------------------------------
    # RECHARGE
    # --------------------------------------------------------

    elif any(word in text for word in [
        "recharge",
        "mobile recharge",
        "airtel recharge",
        "jio recharge",
        "bharti airtel"
    ]):

        return "Recharge"


    # --------------------------------------------------------
    # GOOGLE PAYMENT
    # --------------------------------------------------------

    elif any(word in text for word in [
        "google play",
        "google india",
        "google payment",
        "googlepay",
        "google pay"
    ]):

        return "Google Payment"


    # --------------------------------------------------------
    # INSURANCE
    # --------------------------------------------------------

    elif any(word in text for word in [
        "insurance",
        "star health",
        "lic",
        "policy premium",
        "premium payment"
    ]):

        return "Insurance"


    # --------------------------------------------------------
    # HEALTH
    # --------------------------------------------------------

    elif any(word in text for word in [
        "pharmacy",
        "doctor",
        "health",
        "medicine",
        "medical",
        "hospital",
        "apollo",
        "diagnostic",
        "clinic"
    ]):

        return "Health"


    # --------------------------------------------------------
    # FRIENDS AND FAMILY
    # --------------------------------------------------------

    elif any(word in text for word in [
        "gift",
        "friend",
        "family"
    ]):

        return "Friends & Family"


    # --------------------------------------------------------
    # SALARY
    # --------------------------------------------------------

    elif any(word in text for word in [
        "salary",
        "sal credit",
        "salary credit"
    ]):

        return "Salary"


    # --------------------------------------------------------
    # WALLET / PAYMENT SERVICE
    # --------------------------------------------------------

    elif any(word in text for word in [
        "mobikwik",
        "wallet"
    ]):

        return "Wallet / Payment Service"


    # --------------------------------------------------------
    # MERCHANT PAYMENT
    # --------------------------------------------------------

    elif any(word in text for word in [
        "bharatpe",
        "merchant payment",
        "merchant"
    ]):

        return "Merchant Payment"


    # --------------------------------------------------------
    # CASH WITHDRAWAL
    # --------------------------------------------------------

    elif any(word in text for word in [
        "upicashwdl",
        "cash withdrawal",
        "cashwdl",
        "atm withdrawal",
        "cash withdraw"
    ]):

        return "Cash Withdrawal"


    # --------------------------------------------------------
    # UPI REVERSAL
    # --------------------------------------------------------

    elif any(word in text for word in [
        "upi/rev",
        "upi rev",
        "reversal"
    ]):

        return "UPI Reversal"


    # --------------------------------------------------------
    # BANK TRANSFER
    # --------------------------------------------------------

    elif any(word in text for word in [
        "ift/",
        "neft/",
        "rtgs/",
        "imps/"
    ]):

        return "Bank Transfer"


    # --------------------------------------------------------
    # UPI PAYMENT
    # --------------------------------------------------------

    elif any(word in text for word in [
        "upi/dr",
        "upi payment"
    ]):

        return "UPI Payment"


    # --------------------------------------------------------
    # UPI RECEIPT
    # --------------------------------------------------------

    elif any(word in text for word in [
        "upi/cr",
        "upi receipt"
    ]):

        return "UPI Receipt"


    # --------------------------------------------------------
    # OTHERS
    # --------------------------------------------------------

    else:

        return "Others"


# Apply categories

df["Category"] = df["Transaction Details"].apply(
    assign_category
)


# ============================================================
# 5. CATEGORY DISTRIBUTION
# ============================================================

print("\n")
print("=" * 60)
print("CATEGORY DISTRIBUTION")
print("=" * 60)

print(
    df["Category"].value_counts()
)


# ============================================================
# 6. REMOVE EMPTY TRANSACTION TEXT
# ============================================================

df = df[
    df["Cleaned_Text"].str.strip() != ""
].copy()


# ============================================================
# 7. PREPARE INPUT AND TARGET
# ============================================================

X_text = df["Cleaned_Text"]

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    df["Category"]
)


print("\n")
print("=" * 60)
print("CATEGORY ENCODING")
print("=" * 60)

for i, category in enumerate(
    label_encoder.classes_
):

    print(
        i,
        "=",
        category
    )


# ============================================================
# 8. TRAIN TEST SPLIT
# ============================================================

print("\n")
print("=" * 60)
print("TRAIN TEST SPLIT")
print("=" * 60)


category_counts = df["Category"].value_counts()

print("\nMinimum category count:",
      category_counts.min())


# If every category has at least 2 records,
# stratified split can be used.

if category_counts.min() >= 2:

    X_train_text, X_test_text, y_train, y_test = train_test_split(

        X_text,
        y,

        test_size=0.20,

        random_state=42,

        stratify=y

    )

else:

    print(
        "\nSome categories contain only one transaction."
    )

    print(
        "Using normal train-test split."
    )

    X_train_text, X_test_text, y_train, y_test = train_test_split(

        X_text,
        y,

        test_size=0.20,

        random_state=42

    )


print(
    "\nTraining samples:",
    len(X_train_text)
)

print(
    "Testing samples:",
    len(X_test_text)
)


# ============================================================
# MODEL 1
# LOGISTIC REGRESSION
# ============================================================

print("\n")
print("=" * 60)
print("MODEL 1: LOGISTIC REGRESSION")
print("=" * 60)


tfidf = TfidfVectorizer(

    max_features=5000,

    ngram_range=(1, 2)

)


X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)


logistic_model = LogisticRegression(

    max_iter=1000,

    class_weight="balanced"

)


logistic_model.fit(

    X_train_tfidf,

    y_train

)


logistic_predictions = logistic_model.predict(

    X_test_tfidf

)


logistic_accuracy = accuracy_score(

    y_test,

    logistic_predictions

)


print("\nAccuracy:")

print(

    logistic_accuracy

)


print("\nClassification Report:")

print(

    classification_report(

        y_test,

        logistic_predictions,

        labels=np.arange(
            len(label_encoder.classes_)
        ),

        target_names=label_encoder.classes_,

        zero_division=0

    )

)


# ============================================================
# MODEL 2
# MULTINOMIAL NAIVE BAYES
# ============================================================

print("\n")
print("=" * 60)
print("MODEL 2: MULTINOMIAL NAIVE BAYES")
print("=" * 60)


count_vectorizer = CountVectorizer(

    max_features=5000,

    ngram_range=(1, 2)

)


X_train_count = count_vectorizer.fit_transform(

    X_train_text

)


X_test_count = count_vectorizer.transform(

    X_test_text

)


nb_model = MultinomialNB()


nb_model.fit(

    X_train_count,

    y_train

)


nb_predictions = nb_model.predict(

    X_test_count

)


nb_accuracy = accuracy_score(

    y_test,

    nb_predictions

)


print("\nAccuracy:")

print(

    nb_accuracy

)


print("\nClassification Report:")

print(

    classification_report(

        y_test,

        nb_predictions,

        labels=np.arange(
            len(label_encoder.classes_)
        ),

        target_names=label_encoder.classes_,

        zero_division=0

    )

)


# ============================================================
# MODEL 3
# RANDOM FOREST
# ============================================================

print("\n")
print("=" * 60)
print("MODEL 3: RANDOM FOREST")
print("=" * 60)


rf_model = RandomForestClassifier(

    n_estimators=200,

    random_state=42,

    n_jobs=-1,

    class_weight="balanced"

)


rf_model.fit(

    X_train_tfidf,

    y_train

)


rf_predictions = rf_model.predict(

    X_test_tfidf

)


rf_accuracy = accuracy_score(

    y_test,

    rf_predictions

)


print("\nAccuracy:")

print(

    rf_accuracy

)


print("\nClassification Report:")

print(

    classification_report(

        y_test,

        rf_predictions,

        labels=np.arange(
            len(label_encoder.classes_)
        ),

        target_names=label_encoder.classes_,

        zero_division=0

    )

)


# ============================================================
# MODEL 4
# LSTM
# ============================================================

print("\n")
print("=" * 60)
print("MODEL 4: LSTM")
print("=" * 60)


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = Tokenizer(

    num_words=10000,

    oov_token="<OOV>"

)


tokenizer.fit_on_texts(

    X_train_text

)


X_train_sequences = tokenizer.texts_to_sequences(

    X_train_text

)


X_test_sequences = tokenizer.texts_to_sequences(

    X_test_text

)


# ============================================================
# PADDING
# ============================================================

MAX_LENGTH = 50


X_train_pad = pad_sequences(

    X_train_sequences,

    maxlen=MAX_LENGTH,

    padding="post",

    truncating="post"

)


X_test_pad = pad_sequences(

    X_test_sequences,

    maxlen=MAX_LENGTH,

    padding="post",

    truncating="post"

)


# ============================================================
# NUMBER OF CLASSES
# ============================================================

num_classes = len(

    label_encoder.classes_

)


# ============================================================
# BUILD LSTM
# ============================================================

lstm_model = Sequential()


lstm_model.add(

    Embedding(

        input_dim=10000,

        output_dim=128,

        input_length=MAX_LENGTH

    )

)


lstm_model.add(

    LSTM(128)

)


lstm_model.add(

    Dropout(0.3)

)


lstm_model.add(

    Dense(

        64,

        activation="relu"

    )

)


lstm_model.add(

    Dropout(0.3)

)


lstm_model.add(

    Dense(

        num_classes,

        activation="softmax"

    )

)


# ============================================================
# COMPILE LSTM
# ============================================================

lstm_model.compile(

    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)


# ============================================================
# CONVERT LABELS
# ============================================================

y_train_lstm = to_categorical(

    y_train,

    num_classes=num_classes

)


y_test_lstm = to_categorical(

    y_test,

    num_classes=num_classes

)


# ============================================================
# TRAIN LSTM
# ============================================================

history = lstm_model.fit(

    X_train_pad,

    y_train_lstm,

    epochs=10,

    batch_size=32,

    validation_split=0.10,

    verbose=1

)


# ============================================================
# LSTM PREDICTION
# ============================================================

lstm_predictions_probability = (

    lstm_model.predict(

        X_test_pad

    )

)


lstm_predictions = np.argmax(

    lstm_predictions_probability,

    axis=1

)


lstm_accuracy = accuracy_score(

    y_test,

    lstm_predictions

)


print("\nLSTM Accuracy:")

print(

    lstm_accuracy

)


print("\nLSTM Classification Report:")


print(

    classification_report(

        y_test,

        lstm_predictions,

        labels=np.arange(
            len(label_encoder.classes_)
        ),

        target_names=label_encoder.classes_,

        zero_division=0

    )

)


# ============================================================
# 9. MODEL COMPARISON
# ============================================================

print("\n")
print("=" * 60)
print("MODEL ACCURACY COMPARISON")
print("=" * 60)


results = pd.DataFrame({

    "Model": [

        "Logistic Regression",

        "Multinomial Naive Bayes",

        "Random Forest",

        "LSTM"

    ],

    "Accuracy": [

        logistic_accuracy,

        nb_accuracy,

        rf_accuracy,

        lstm_accuracy

    ]

})


results["Accuracy (%)"] = (

    results["Accuracy"] * 100

).round(2)


print("\n")

print(

    results.to_string(

        index=False

    )

)


# ============================================================
# 10. FIND BEST MODEL
# ============================================================

best_model = results.loc[

    results["Accuracy"].idxmax()

]


print("\n")
print("=" * 60)
print("BEST MODEL")
print("=" * 60)


print(

    "Best Model:",

    best_model["Model"]

)


print(

    "Accuracy:",

    best_model["Accuracy (%)"],

    "%"

)


# ============================================================
# 11. SAVE MODEL COMPARISON
# ============================================================

results.to_csv(

    "model_comparison.csv",

    index=False

)


# ============================================================
# 12. SAVE CATEGORIZED DATASET
# ============================================================

df.to_csv(

    "categorized_transactions.csv",

    index=False

)


print("\n")
print("=" * 60)
print("FILES SAVED")
print("=" * 60)

print(

    "1. model_comparison.csv"

)

print(

    "2. categorized_transactions.csv"

)

print("\nProgram completed successfully!")

Dataset loaded successfully!
Total transactions: 649

Columns:
['Date', 'Value Date', 'Transaction Details', 'Debit', 'Credit', 'Balance']


CATEGORY DISTRIBUTION
Category
Others                      555
Food                         27
Shopping                     13
Bank Transfer                12
Cash Withdrawal              11
Travel                        7
Entertainment                 6
Salary                        5
Wallet / Payment Service      4
Bills                         4
UPI Reversal                  3
Merchant Payment              1
Insurance                     1
Name: count, dtype: int64


CATEGORY ENCODING
0 = Bank Transfer
1 = Bills
2 = Cash Withdrawal
3 = Entertainment
4 = Food
5 = Insurance
6 = Merchant Payment
7 = Others
8 = Salary
9 = Shopping
10 = Travel
11 = UPI Reversal
12 = Wallet / Payment Service


TRAIN TEST SPLIT

Minimum category count: 1

Some categories contain only one transaction.
Using normal train-test split.

Training samples: 519
Testing sample